In [119]:
import pandas as pd
import os
import pickle

# SETS

In [120]:
# Define the directory containing the CSV files
CLEWsy_outputs_dir= 'datapackage/SETS/CLEWsy_outputs'

In [121]:
# Get a list of all CSV files in the directory
SETS_csv_files = [f for f in os.listdir(CLEWsy_outputs_dir) if f.endswith('.csv')]

# Create a dataframe for each CSV file with the corresponding name
SETS = {}

for csv_file in SETS_csv_files:
    # Strip the .csv extension to get the dataframe name
    df_name = os.path.splitext(csv_file)[0]
    
    # Read the CSV file into a dataframe
    df_path = os.path.join(CLEWsy_outputs_dir, csv_file)
    SETS[df_name] = pd.read_csv(df_path)
    
    # Optionally, you can print the dataframe name to verify
    print(f"SETS Dataframe created: {df_name}")
print(f"\nTo access the SETS dataframe, type >>> SETS['set_name_str']")

SETS Dataframe created: COMMODITY
SETS Dataframe created: STORAGE
SETS Dataframe created: MODE_OF_OPERATION
SETS Dataframe created: TIMESLICE
SETS Dataframe created: OutputActivityRatio
SETS Dataframe created: SEASON
SETS Dataframe created: DAILYTIMEBRACKET
SETS Dataframe created: InputActivityRatio
SETS Dataframe created: FUEL
SETS Dataframe created: DAYTYPE
SETS Dataframe created: EMISSION
SETS Dataframe created: YEAR
SETS Dataframe created: REGION
SETS Dataframe created: TECHNOLOGY

To access the SETS dataframe, type >>> SETS['set_name_str']


# PARAMETERS

In [122]:
# Define the directory containing the CSV files
REF_parameter_files_dir= 'datapackage/Params'

In [123]:
# Get a list of all CSV files in the directory
Params_csv_files = [f for f in os.listdir(REF_parameter_files_dir) if f.endswith('.csv')]

# Create a dataframe for each CSV file with the corresponding name
Parameters = {}

for csv_file in Params_csv_files:
    # Strip the .csv extension to get the dataframe name
    df_name = os.path.splitext(csv_file)[0]
    
    # Read the CSV file into a dataframe
    df_path = os.path.join(REF_parameter_files_dir, csv_file)
    Parameters[df_name] = pd.read_csv(df_path)
    
    # Optionally, you can print the dataframe name to verify
    print(f"Parameters Dataframe created: {df_name}")
print(f"\nTo access the Parameters dataframe, type >>> Parameters['set_name_str']")

Parameters Dataframe created: ReserveMarginTagFuel
Parameters Dataframe created: DaySplit
Parameters Dataframe created: REMinProductionTarget
Parameters Dataframe created: RETagFuel
Parameters Dataframe created: ModelPeriodEmissionLimit
Parameters Dataframe created: MinStorageCharge
Parameters Dataframe created: OperationalLifeStorage
Parameters Dataframe created: TradeRoute
Parameters Dataframe created: TotalAnnualMaxCapacityInvestment
Parameters Dataframe created: DiscountRateStorage
Parameters Dataframe created: TechnologyToStorage
Parameters Dataframe created: DaysInDayType
Parameters Dataframe created: CapitalCostStorage
Parameters Dataframe created: OperationalLife
Parameters Dataframe created: TotalTechnologyAnnualActivityLowerLimit
Parameters Dataframe created: Conversionls
Parameters Dataframe created: ReserveMargin
Parameters Dataframe created: StorageMaxDischargeRate
Parameters Dataframe created: CapitalCost
Parameters Dataframe created: TechnologyActivityByModeLowerLimit
Pa

In [124]:
Parameters['CapacityFactor']

,Unnamed: 0,REGION,TECHNOLOGY,TIMESLICE,YEAR,VALUE
0,0,REGION1,DEMAGRDSL,SEA1D,2020,1.000000
1,1,REGION1,DEMAGRDSL,SEA1N,2020,1.000000
2,2,REGION1,DEMAGRDSL,SEA2D,2020,1.000000
3,3,REGION1,DEMAGRDSL,SEA2N,2020,1.000000
4,4,REGION1,DEMAGRDSL,SEA3D,2020,1.000000
...,...,...,...,...,...,...
39347,39347,REGION1,PWRSOLB03,SEA2N,2050,0.670939
39348,39348,REGION1,PWRSOLB03,SEA3D,2050,0.506740
39349,39349,REGION1,PWRSOLB03,SEA3N,2050,0.604514
39350,39350,REGION1,PWRSOLB03,SEA4D,2050,0.487323


In [125]:
from pydantic import BaseModel, Field

class CapacityFactor(BaseModel):
    REGION: str
    TECHNOLOGY: str
    TIMESLICE: str
    YEAR: int
    VALUE: float = Field(..., alias='VALUE', env='CAPACITY_FACTOR_VALUE', ge=0, le=1)


In [126]:
import pandas as pd

# Read the existing dataset from the CSV file
solar_sites = pd.read_csv('from_linking_tool/new_sites_timeslice/Solar_East Kootenay_1.csv')

solar_sites=solar_sites.iloc[:, :2]

In [127]:
wind_ts_dir='from_linking_tool/wind_ts.pkl'
# Load the dictionary from the pickle file
with open(wind_ts_dir, 'rb') as file:
    wind_ts = pickle.load(file)

In [129]:
for site in wind_ts.keys():
    # Convert DataFrame to list of dictionaries
    _data_dict_ = wind_ts[site].to_dict(orient='records')

    # Validate the row using the Pydantic model
    rows = [CapacityFactor(**item)for item in _data_dict_]

_rows_dumped_ = [row.model_dump() for row in rows]
# Append the validated row to the dataframe
_data_ = pd.concat([Parameters['CapacityFactor'], pd.DataFrame(_rows_dumped_)], ignore_index=True)
Parameters['CapacityFactor'] = _data_.loc[:, ['REGION', 'TECHNOLOGY', 'YEAR', 'TIMESLICE', 'VALUE']]

In [130]:
Parameters['CapacityFactor']

,REGION,TECHNOLOGY,YEAR,TIMESLICE,VALUE
0,REGION1,DEMAGRDSL,2020,SEA1D,1.000000
1,REGION1,DEMAGRDSL,2020,SEA1N,1.000000
2,REGION1,DEMAGRDSL,2020,SEA2D,1.000000
3,REGION1,DEMAGRDSL,2020,SEA2N,1.000000
4,REGION1,DEMAGRDSL,2020,SEA3D,1.000000
...,...,...,...,...,...
39515,REGION1,PWRSOLB03,2050,SEA2N,0.670939
39516,REGION1,PWRSOLB03,2050,SEA3D,0.506740
39517,REGION1,PWRSOLB03,2050,SEA3N,0.604514
39518,REGION1,PWRSOLB03,2050,SEA4D,0.487323


In [ ]:
# Define the directory containing the CSV files
REF_parameter_updated_files_dir= 'datapackage/updated_data_test'
Parameters['CapacityFactor'].to_csv(os.path.join(REF_parameter_updated_files_dir,f"CapacityFactor.csv"))